In [1]:
!uv pip install openai tiktoken pydantic tenacity

Using Python 3.11.15 environment at: /Users/furqat/Desktop/ai-engineering/.venv
Checked 4 packages in 34ms


In [ ]:
from openai import OpenAI

In [18]:

client = OpenAI()
MODEL = "gpt-6-luna"

In [19]:
from openai.types import model
responses = client.responses.create(
    model=MODEL,
    input="salom"
)


In [ ]:
print(responses.model_dump_json(indent=2))

In [15]:
history = []

In [16]:
def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    res = client.responses.create(
        model=MODEL,
        instructions="You are a gamer",
        input=history
    )
    history.append({"role": "assistant", "content": res.output_text})

    return res.output_text

In [ ]:
print(chat("Salom ismim Furqat"))
print(chat("ismim nima?"))

In [22]:
#temperature
from openai import BadRequestError

try:
    r = client.responses.create(
        model=MODEL,
        temperature=0.2,
        input="Salom"
    )
    print("ishladi: ", r.output_text)
except BadRequestError as e:
    print("ishlamadi: ", e.message)

ishlamadi:  Error code: 400 - Unsupported parameter: 'temperature' is not supported with this model.


In [27]:
#max tokens
r = client.responses.create(
    model=MODEL,
    input="Uzbeksitan tarixi xaqida yoz",
    max_output_tokens=3000
)

print(r.status)
print(r.incomplete_details)
print(repr(r.output_text))

completed
None
'## O‘zbekiston tarixi\n\nO‘zbekiston hududi qadimdan Markaziy Osiyoning muhim sivilizatsiya va savdo markazlaridan biri bo‘lib kelgan. Bu yerda dehqonchilik, hunarmandchilik va shahar madaniyati rivojlangan, Buyuk ipak yo‘li esa Sharq va G‘arbni bog‘lagan.\n\n### Qadimgi davr\n\nHozirgi O‘zbekiston hududida qadimda **So‘g‘diyona, Baqtriya va Xorazm** kabi tarixiy o‘lkalar mavjud bo‘lgan. Miloddan avvalgi VI–IV asrlarda bu yerlarning ayrim qismlari Ahamoniylar imperiyasi tarkibiga kirgan. Miloddan avvalgi IV asrda Aleksandr Makedonskiy Markaziy Osiyoga yurish qilgan. Keyingi asrlarda hududda Yunon-Baqtriya va Kushon podsholiklari kabi davlatlar hukmronlik qilgan.\n\n### O‘rta asrlar\n\nVIII asrda arablar Movarounnahrni egallab, islom dini keng tarqala boshlagan. IX–X asrlarda Buxoroni markaz qilgan **Somoniylar davlati** yuksaldi. Bu davrda ilm-fan, adabiyot va me’morchilik rivojlandi. Keyinchalik hudud Qoraxoniylar va Xorazmshohlar kabi davlatlar tarkibida bo‘ldi.\n\nXI

In [29]:
import tiktoken

enc = tiktoken.get_encoding('o200k_base')

for text in ["How are you?", "Qalaysan, ishlaring yaxshimi?", "Как дела?"]:
    tokens = enc.encode(text)
    print(len(tokens), text)
    print([enc.decode([t]) for t in tokens])

4 How are you?
['How', ' are', ' you', '?']
10 Qalaysan, ishlaring yaxshimi?
['Q', 'al', 'aysan', ',', ' ish', 'laring', ' yax', 'sh', 'imi', '?']
3 Как дела?
['Как', ' дела', '?']


In [30]:
PRICE_IN = 0.10
PRICE_OUT = 0.50

def cost_usd(usage) -> float:
    return (usage.input_tokens * PRICE_IN + usage.output_tokens * PRICE_OUT) / 1_000_000

r = client.responses.create(
    model=MODEL,
    input="Rekursiya nima?"
)

print(r.usage.input_tokens, r.usage.output_tokens, f"{cost_usd(r.usage):.6f}")

11 173 0.000088


In [31]:
#error handling
from openai import NotFoundError, AuthenticationError

try:
    OpenAI(api_key="sk-worng").responses.create(model=MODEL, input="Salom")
except AuthenticationError as e:
    print("Xatolik: ", e.message)

try:
    client.responses.create(model="wrong-model", input="salom")
except NotFoundError as e:
    print("Xatolik: ", e.message)

Xatolik:  Error code: 401 - Incorrect API key provided: sk-worng. You can find your API key at https://platform.openai.com/account/api-keys.
Xatolik:  Error code: 404 - The model `wrong-model` does not exist or you do not have access to it.


In [33]:
#streaming
stream = client.responses.create(
    model=MODEL,
    input="Top features of python",
    stream=True
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print("\n\nTugadi")

### Top features of Python

- **Readable syntax:** Its clear, concise style makes code easier to write and understand.
- **Easy to learn:** Python is often recommended for beginners.
- **Interpreted:** You can run code without a separate compilation step.
- **Dynamically typed:** Variables don’t need explicit type declarations.
- **Multi-paradigm:** Supports procedural, object-oriented, and functional programming.
- **Large standard library:** Includes tools for tasks such as file handling, networking, and data processing.
- **Extensive ecosystem:** Many third-party packages support areas like web development, automation, data science, and AI.
- **Cross-platform:** Python programs can run on Windows, macOS, and Linux.
- **Automatic memory management:** Python handles much of the memory allocation and cleanup for you.
- **Extensible:** It can work with code and libraries written in languages such as C and C++.

Tugadi


In [45]:
from openai.types.decision_create_params import QuestionQuestionParamPredicate
#structured output
from pydantic import BaseModel, Field
from typing import Literal

class QuizQuestion(BaseModel):
    question: str
    options: list[str] = Field(description="4 ta variant")
    correct_index: int = Field(ge=0, le=3)
    explanation: str

class Quiz(BaseModel):
    topic: str
    difficulty: Literal["easy", "medium", "hard"]
    questions: list[QuizQuestion]

res = client.responses.parse(
        model=MODEL,
        instructions="Berilgan matn asosida o'zbek tilida test tuz. 5 ta",
        input="Python'da list o'zgaruvchan (mutable), tuple esa o'zgarmas (immutable). ...",
        text_format=Quiz
    )

quiz = res.output_parsed

print(quiz.topic, quiz.difficulty)

for q in quiz.questions:
    print(q.question, "->", q.options[q.correct_index])

Python’da list va tuple easy
Python’da qaysi ma’lumot turi o‘zgaruvchan (mutable) hisoblanadi? -> list
Python’da qaysi ma’lumot turi o‘zgarmas (immutable) hisoblanadi? -> tuple
List va tuple o‘rtasidagi asosiy farq nima? -> List o‘zgaruvchan, tuple o‘zgarmas
Agar yaratilgandan keyin elementlarini o‘zgartirish kerak bo‘lsa, qaysi tur mos keladi? -> list
Qaysi jumla to‘g‘ri? -> List o‘zgaruvchan, tuple o‘zgarmas
